# Regression Project — Data Split, EDA & Preprocessing

**Team Member Responsibility:** EDA + Preprocessing  
**Dataset:** Student Performance Dataset  
**Target Variable (Y):** `FinalGrade`

---

## Notebook Structure
1. Load Data & Initial Inspection
2. **Train/Test Split FIRST** (before any preprocessing)
3. Exploratory Data Analysis (EDA) — on Train only
4. Preprocessing Pipeline — fit on Train, transform both
5. Save cleaned datasets for the modeling team

## Cell 1 — Import Libraries

We import all necessary libraries upfront.
- `pandas` / `numpy`: data manipulation
- `matplotlib` / `seaborn`: visualization
- `plotly`: interactive visualizations (required by project spec)
- We do **NOT** import sklearn for modeling — only for the split utility, which is allowed since it is not used for regression math.

In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')


print('Libraries loaded successfully.')

Libraries loaded successfully.


## Cell 2 — Load Raw Data & Initial Inspection

We load the raw dataset and take a first look **without modifying anything yet**.  
The goal here is to understand:
- How many rows and columns we have
- What data types each column has
- Whether there are obvious missing values or issues

> We do NOT clean or modify anything in this step. We observe only.

In [51]:
# Load the raw dataset
df_raw = pd.read_csv('regression.csv')

print('=' * 50)
print(f'Dataset Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
print('=' * 50)

print('\nColumn Names and Data Types:')
print(df_raw.dtypes)

print('\nFirst 5 rows:')
display(df_raw.head())

Dataset Shape: 1000 rows x 12 columns

Column Names and Data Types:
StudentID                    float64
Name                          object
Gender                        object
AttendanceRate               float64
StudyHoursPerWeek            float64
PreviousGrade                float64
ExtracurricularActivities    float64
ParentalSupport               object
FinalGrade                   float64
Study Hours                  float64
Attendance (%)               float64
Online Classes Taken          object
dtype: object

First 5 rows:


,StudentID,Name,Gender,AttendanceRate,StudyHoursPerWeek,PreviousGrade,ExtracurricularActivities,ParentalSupport,FinalGrade,Study Hours,Attendance (%),Online Classes Taken
0,1.0,John,Male,85.0,15.0,78.0,1.0,High,80.0,4.8,59.0,False
1,2.0,Sarah,Female,90.0,20.0,85.0,2.0,Medium,87.0,2.2,70.0,True
2,3.0,Alex,Male,78.0,10.0,65.0,0.0,Low,68.0,4.6,92.0,False
3,4.0,Michael,Male,92.0,25.0,90.0,3.0,High,92.0,2.9,96.0,False
4,5.0,Emma,Female,NaN,18.0,82.0,2.0,Medium,85.0,4.1,97.0,True


## Cell 3 — Identify Messy Issues (Before Split)

Before splitting, we do a **quick diagnosis** to document all messiness in the dataset.  
This is observation only — we note every problem but fix nothing yet.

Issues we look for:
1. Missing values
2. Duplicate or redundant columns
3. Outliers / impossible values
4. Categorical columns needing encoding
5. Irrelevant columns (identifiers)

In [52]:
# --- 1. Missing Values ---
print('=== MISSING VALUES ===')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

# --- 2. Redundant Column Check ---
# AttendanceRate vs Attendance(%) -- different ranges despite measuring same concept
# StudyHoursPerWeek vs Study Hours -- different scales for same concept
print('\n=== REDUNDANT COLUMN PAIRS ===')
print('AttendanceRate  range:', df_raw['AttendanceRate'].min(), '-', df_raw['AttendanceRate'].max())
print('Attendance (%)  range:', df_raw['Attendance (%)'].min(), '-', df_raw['Attendance (%)'].max())
print('StudyHoursPerWeek range:', df_raw['StudyHoursPerWeek'].min(), '-', df_raw['StudyHoursPerWeek'].max())
print('Study Hours       range:', df_raw['Study Hours'].min(), '-', df_raw['Study Hours'].max())

print("Correlation between StudyHoursPerWeek and Study Hours:")
print(df_raw[['StudyHoursPerWeek', 'Study Hours']].corr())
print()

print("Correlation between AttendanceRate and Attendance (%):")
print(df_raw[['AttendanceRate', 'Attendance (%)']].corr())

# --- 3. Impossible Outliers ---
print('\n=== IMPOSSIBLE VALUES (Attendance > 100%) ===')
print('Attendance (%) > 100 count:', (df_raw['Attendance (%)'] > 100).sum())
print('Values found:', df_raw[df_raw['Attendance (%)'] > 100]['Attendance (%)'].unique())

# --- 4. Categorical Columns ---
print('\n=== CATEGORICAL COLUMNS ===')
print('Gender unique           :', df_raw['Gender'].unique())
print('ParentalSupport unique  :', df_raw['ParentalSupport'].unique())
print('Online Classes Taken    :', df_raw['Online Classes Taken'].unique())

# --- 5. Irrelevant Columns ---
print('\n=== IRRELEVANT COLUMNS ===')
print('StudentID -- unique identifier, no predictive signal')
print('Name      -- free text identifier, should be dropped')

=== MISSING VALUES ===
                           Missing Count  Missing %
StudentID                             40        4.0
Name                                  34        3.4
Gender                                48        4.8
AttendanceRate                        40        4.0
StudyHoursPerWeek                     50        5.0
PreviousGrade                         33        3.3
ExtracurricularActivities             43        4.3
ParentalSupport                       22        2.2
FinalGrade                            40        4.0
Study Hours                           24        2.4
Attendance (%)                        41        4.1
Online Classes Taken                  25        2.5

=== REDUNDANT COLUMN PAIRS ===
AttendanceRate  range: 70.0 - 95.0
Attendance (%)  range: 50.0 - 200.0
StudyHoursPerWeek range: 8.0 - 30.0
Study Hours       range: -5.0 - 5.0
Correlation between StudyHoursPerWeek and Study Hours:
                   StudyHoursPerWeek  Study Hours
StudyHoursPerWeek    

### Summary of Issues Found

| Issue | Columns Affected | Action Planned |
|---|---|---|
| Missing values (~3-5%) | All columns | Median imputation for numeric, Mode for categorical |
| Redundant columns | `AttendanceRate` vs `Attendance (%)` | Keep `Attendance (%)`, drop `AttendanceRate` |
| Redundant columns | `StudyHoursPerWeek` vs `Study Hours` | Keep `StudyHoursPerWeek`, drop `Study Hours` |
| Impossible outlier | `Attendance (%)` = 200 | Cap at 100 (physical maximum) |
| Categorical encoding | `Gender`, `ParentalSupport`, `Online Classes Taken` | Binary / Ordinal Encoding |
| Irrelevant identifiers | `StudentID`, `Name` | Drop entirely |

---
## Cell 4 — Train / Test Split

### Why split BEFORE preprocessing? (The most important rule)

> **The test set must simulate unseen future data.  
> If we preprocess using information from the test set, we are cheating — this is called Data Leakage.**

#### Concrete example of Data Leakage:
If we fill missing values with the **median of the whole dataset** (including test):  
- The test median leaks into our training imputation  
- Our model performs artificially well on test — but will fail on truly new data

#### The correct workflow:
```
Raw Data
   |
   |---- Train (80%) --> fit imputer/scaler on this --> transform train
   |                                                         |
   |---- Test  (20%) -----------------------------> transform test (using train stats only)
```

#### Do we split more than once?
**No.** We split once here into Train and Test.  
Cross-Validation (K-Fold) happens **entirely inside the Train set** — handled by the modeling team.  
The Test set is **locked** and never touched again until final evaluation.

```
Train Set (80%)
   |
   |-- Fold 1: [VAL | TRAIN | TRAIN | TRAIN | TRAIN]
   |-- Fold 2: [TRAIN | VAL | TRAIN | TRAIN | TRAIN]   <- Cross Validation (modeling team's job)
   |-- Fold 3: [TRAIN | TRAIN | VAL | TRAIN | TRAIN]
   |-- Fold 4: [TRAIN | TRAIN | TRAIN | VAL | TRAIN]
   |-- Fold 5: [TRAIN | TRAIN | TRAIN | TRAIN | VAL]

Test Set (20%) -----------------------> LOCKED. Touch only at the very end.
```

In [53]:
# Set random seed for reproducibility
# VERY important: every team member must use this same seed
# to get the exact same train/test split
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [54]:
# Manual Train/Test split using NumPy — implemented from scratch

# Step 1: Create array of all row indices and shuffle randomly
n = len(df_raw)
indices = np.arange(n)          # array: [0, 1, 2, ..., 999]
np.random.shuffle(indices)      # shuffle in-place using fixed RANDOM_SEED

# Step 2: Define split point (80% train, 20% test)
TRAIN_RATIO = 0.80
split_point = int(n * TRAIN_RATIO)   # 1000 * 0.80 = 800

# Step 3: Slice into train and test index arrays
train_indices = indices[:split_point]   # first 800 shuffled indices
test_indices  = indices[split_point:]   # remaining 200 shuffled indices

# Step 4: Build the two DataFrames
df_train = df_raw.iloc[train_indices].reset_index(drop=True)
df_test  = df_raw.iloc[test_indices].reset_index(drop=True)

print(f'Total samples    : {n}')
print(f'Train set size   : {len(df_train)} ({len(df_train)/n*100:.0f}%)')
print(f'Test  set size   : {len(df_test)}  ({len(df_test)/n*100:.0f}%)')
print(f'Random Seed used : {RANDOM_SEED}  <- every team member must use this same seed!')

# Sanity check: confirm no index overlap between train and test
overlap = set(train_indices) & set(test_indices)
print(f'\nIndex overlap between train and test: {len(overlap)} (must be 0)')

Total samples    : 1000
Train set size   : 800 (80%)
Test  set size   : 200  (20%)
Random Seed used : 42  <- every team member must use this same seed!

Index overlap between train and test: 0 (must be 0)


---
## Cell 5 — EDA: Target Variable

> All EDA is performed on the **TRAIN SET only**.  
> The test set is set aside and will not be examined until final evaluation.

### Why inspect the target variable first?
Linear regression assumes the target Y follows an approximately normal distribution.  
If Y is heavily skewed, we may need a transformation (e.g., log). We check this here.

In [55]:
# Analyze the target variable -- FinalGrade (train set only)
target_series = df_train['FinalGrade'].dropna()

print('=== FinalGrade Statistics (Train Set) ===')
print(f'Count  : {target_series.count()}')
print(f'Mean   : {target_series.mean():.2f}')
print(f'Median : {target_series.median():.2f}')
print(f'Std    : {target_series.std():.2f}')
print(f'Min    : {target_series.min()}')
print(f'Max    : {target_series.max()}')

# Interactive histogram
fig = px.histogram(
    df_train.dropna(subset=['FinalGrade']),
    x='FinalGrade',
    nbins=20,
    title='Distribution of FinalGrade (Train Set)',
    labels={'FinalGrade': 'Final Grade'},
    color_discrete_sequence=['steelblue']
)
fig.add_vline(
    x=target_series.mean(), line_dash='dash', line_color='red',
    annotation_text=f'Mean: {target_series.mean():.1f}'
)
fig.update_layout(bargap=0.1)
fig.show()

=== FinalGrade Statistics (Train Set) ===
Count  : 768
Mean   : 79.91
Median : 80.00
Std    : 9.63
Min    : 62.0
Max    : 92.0


## Cell 6 — EDA: Missing Values Visualization

A bar chart of missing percentages per column helps us decide:
- Which columns have the most missingness
- Whether the missing rate is low enough for simple imputation (< 20% is generally safe)

In [56]:
# Visualize missing values on the train set
missing_train = df_train.isnull().sum()
missing_train_pct = (missing_train / len(df_train) * 100).round(2)

fig = px.bar(
    x=missing_train_pct.index,
    y=missing_train_pct.values,
    title='Missing Values per Column (Train Set, %)',
    labels={'x': 'Column', 'y': 'Missing (%)'},
    color=missing_train_pct.values,
    color_continuous_scale='Reds'
)
fig.update_layout(xaxis_tickangle=-40)
fig.show()

print('Missing % per column (Train):')
print(missing_train_pct[missing_train_pct > 0])

Missing % per column (Train):
StudentID                    3.38
Name                         3.62
Gender                       4.00
AttendanceRate               4.00
StudyHoursPerWeek            4.75
PreviousGrade                2.88
ExtracurricularActivities    4.38
ParentalSupport              2.38
FinalGrade                   4.00
Study Hours                  2.25
Attendance (%)               4.25
Online Classes Taken         2.75
dtype: float64


## Cell 7 — EDA: Correlation Heatmap

The correlation heatmap shows the **linear relationship** between all numeric variables.

Key things to look for:
- **High correlation with FinalGrade** → strong predictors (absolute value closer to 1)
- **High correlation between two predictors** → multicollinearity (may need to drop one)
- Values range from -1 (perfect negative) to +1 (perfect positive)

In [57]:
# Compute correlation matrix on numeric columns of train set only
num_for_corr = ['AttendanceRate', 'StudyHoursPerWeek', 'PreviousGrade',
                'ExtracurricularActivities', 'Attendance (%)', 'Study Hours', 'FinalGrade']

corr_df = df_train[num_for_corr].apply(pd.to_numeric, errors='coerce').corr()

fig = px.imshow(
    corr_df,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation Heatmap -- Numeric Features (Train Set)'
)
fig.update_layout(width=700, height=600)
fig.show()

# Print correlation with target specifically
print('\nCorrelation with FinalGrade (sorted by absolute value):')
target_corr = corr_df['FinalGrade'].drop('FinalGrade').sort_values(key=abs, ascending=False)
print(target_corr)


Correlation with FinalGrade (sorted by absolute value):
ExtracurricularActivities   -0.049999
Study Hours                  0.048666
Attendance (%)               0.037202
StudyHoursPerWeek            0.005516
AttendanceRate              -0.004138
PreviousGrade                0.003981
Name: FinalGrade, dtype: float64


## Cell 8 — EDA: Box Plots of Numeric Features

Box plots show the spread, median, and outliers of each numeric variable.  
This helps us confirm which features need outlier treatment.

In [58]:
# Box plots for numeric features on train set
numeric_cols_plot = ['StudyHoursPerWeek', 'PreviousGrade', 'ExtracurricularActivities',
                     'Attendance (%)', 'FinalGrade']

fig = make_subplots(rows=1, cols=len(numeric_cols_plot),
                    subplot_titles=numeric_cols_plot)

for i, col in enumerate(numeric_cols_plot, 1):
    data = pd.to_numeric(df_train[col], errors='coerce').dropna()
    fig.add_trace(
        go.Box(y=data, name=col, showlegend=False),
        row=1, col=i
    )

fig.update_layout(
    title_text='Box Plots of Numeric Features (Train Set)',
    height=450, width=1000
)
fig.show()

## Cell 9 — EDA: Categorical Features vs FinalGrade

We check whether categorical features have any relationship with the target.  
If mean FinalGrade is similar across all categories, the feature may not be useful.

In [59]:
# Mean FinalGrade by categorical features
cat_cols = ['Gender', 'ParentalSupport']

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=[f'Mean FinalGrade by {c}' for c in cat_cols])

for i, col in enumerate(cat_cols, 1):
    group = (
        df_train.groupby(col)['FinalGrade']
        .mean()
        .reset_index()
        .dropna()
    )
    fig.add_trace(
        go.Bar(x=group[col], y=group['FinalGrade'].round(2),
               name=col, showlegend=False,
               text=group['FinalGrade'].round(2), textposition='outside'),
        row=1, col=i
    )

fig.update_layout(
    title='Mean FinalGrade by Categorical Features (Train Set)',
    height=400, yaxis_range=[0, 100], yaxis2_range=[0, 100]
)
fig.show()

##"Categorical features such as Gender and ParentalSupport show very small differences in mean FinalGrade, suggesting weak predictive power."

## Cell 10 — EDA: Scatter Plot — PreviousGrade vs FinalGrade

Based on the correlation heatmap, PreviousGrade is likely the strongest predictor.  
A scatter plot confirms whether the relationship is linear, which is the core assumption of our model.

In [60]:
# Scatter: strongest predictor vs target
fig = px.scatter(
    df_train.dropna(subset=['PreviousGrade', 'FinalGrade']),
    x='PreviousGrade',
    y='FinalGrade',
    color='Gender',
    trendline='ols',
    title='PreviousGrade vs FinalGrade (Train Set)',
    opacity=0.6,
    labels={'PreviousGrade': 'Previous Grade', 'FinalGrade': 'Final Grade'}
)
fig.show()

---
## Cell 11 — Preprocessing: Drop Irrelevant & Redundant Columns

**Justification for each dropped column:**

- `StudentID`: A unique identifier — carries no information about academic performance
- `Name`: Free-text name — identifier only, not a feature
- `AttendanceRate` vs `Attendance (%)`: Both columns measure attendance but on different scales with near-zero correlation (r ≈ 0.01), meaning they are NOT equivalent. We keep `Attendance (%)` as it uses the standard percentage scale (0–100)
- `StudyHoursPerWeek` vs `Study Hours`: Both measure study time but on different scales (weekly hours vs daily-ish hours), correlation ≈ -0.02. We keep `StudyHoursPerWeek` as weekly hours is a more standard academic metric

In [61]:
# Drop irrelevant and redundant columns from both train and test
cols_to_drop = ['StudentID', 'Name', 'AttendanceRate', 'Study Hours']

train = df_train.drop(columns=cols_to_drop).copy()
test  = df_test.drop(columns=cols_to_drop).copy()

print('Remaining columns after dropping:')
print(train.columns.tolist())
print(f'\nTrain shape: {train.shape}')
print(f'Test  shape: {test.shape}')

Remaining columns after dropping:
['Gender', 'StudyHoursPerWeek', 'PreviousGrade', 'ExtracurricularActivities', 'ParentalSupport', 'FinalGrade', 'Attendance (%)', 'Online Classes Taken']

Train shape: (800, 8)
Test  shape: (200, 8)


## Cell 12 — Preprocessing: Fix Outliers

**Outlier in `Attendance (%)`:**
Some rows have `Attendance (%) = 200`, which is physically impossible.  
This is clearly a data entry error (200 instead of 100, or a duplication bug).  

**Fix:** Cap the value at 100 using `clip(upper=100)`.  
This is a **domain-knowledge rule**, not a statistical one — so we apply it to both sets without leakage concern.

In [62]:
# Fix impossible attendance values by capping at 100
print('Before fix:')
print(f'  Train -- rows with Attendance > 100: {(train["Attendance (%)"] > 100).sum()}')
print(f'  Test  -- rows with Attendance > 100: {(test["Attendance (%)"] > 100).sum()}')

# Apply the cap (domain rule: attendance cannot exceed 100%)
train['Attendance (%)'] = train['Attendance (%)'].clip(upper=100)
test['Attendance (%)']  = test['Attendance (%)'].clip(upper=100)

print('\nAfter fix:')
print(f'  Train -- rows with Attendance > 100: {(train["Attendance (%)"] > 100).sum()}')
print(f'  Test  -- rows with Attendance > 100: {(test["Attendance (%)"] > 100).sum()}')

Before fix:
  Train -- rows with Attendance > 100: 8
  Test  -- rows with Attendance > 100: 2

After fix:
  Train -- rows with Attendance > 100: 0
  Test  -- rows with Attendance > 100: 0


## Cell 13 — Preprocessing: Impute Missing Values

### The Golden Rule:
> Compute all imputation statistics (median, mode) from the **Train set only**.  
> Then apply those same statistics to fill missing values in the **Test set**.

**Why?**  
If we used test data to compute the median, we are letting the test set influence our model's training data — that is Data Leakage.

**Strategy:**
- **Numeric columns** → fill with **median** (median is robust to the outliers we just fixed)
- **Categorical columns** → fill with **mode** (most frequent category)
- **FinalGrade (target)** → drop rows where Y is missing (we cannot train or evaluate without it)

In [63]:
# Define column groups
categorical_cols = ['Gender', 'ParentalSupport']
bool_col         = 'Online Classes Taken'
target_col       = 'FinalGrade'
numeric_cols     = [c for c in train.columns
                    if c not in categorical_cols + [bool_col, target_col]]

print('Numeric columns :', numeric_cols)
print('Categorical cols:', categorical_cols)
print('Boolean column  :', bool_col)

# --- Compute imputation values from TRAIN only ---
train_medians = {}
train_modes   = {}

print('\n--- Imputing Numeric Columns with TRAIN Median ---')
for col in numeric_cols:
    train_medians[col] = train[col].median()            # compute from train
    train[col] = train[col].fillna(train_medians[col])  # apply to train
    test[col]  = test[col].fillna(train_medians[col])   # apply TRAIN median to test!
    print(f'  {col}: median = {train_medians[col]:.2f}')

print('\n--- Imputing Categorical Columns with TRAIN Mode ---')
for col in categorical_cols:
    train_modes[col] = train[col].mode()[0]             # compute from train
    train[col] = train[col].fillna(train_modes[col])    # apply to train
    test[col]  = test[col].fillna(train_modes[col])     # apply TRAIN mode to test!
    print(f'  {col}: mode = {train_modes[col]}')

print('\n--- Imputing Boolean Column ---')
bool_mode = train[bool_col].mode()[0]
train[bool_col] = train[bool_col].fillna(bool_mode)
test[bool_col]  = test[bool_col].fillna(bool_mode)
print(f'  {bool_col}: mode = {bool_mode}')

# Drop rows where the target Y is missing (we cannot learn or evaluate without it)
before_train = len(train)
before_test  = len(test)
train = train.dropna(subset=[target_col])
test  = test.dropna(subset=[target_col])
print(f'\nDropped {before_train - len(train)} train rows with missing FinalGrade')
print(f'Dropped {before_test  - len(test)}  test  rows with missing FinalGrade')

print(f'\nFinal shapes -- Train: {train.shape}, Test: {test.shape}')
print('Remaining missing in train:', train.isnull().sum().sum())
print('Remaining missing in test :', test.isnull().sum().sum())

Numeric columns : ['StudyHoursPerWeek', 'PreviousGrade', 'ExtracurricularActivities', 'Attendance (%)']
Categorical cols: ['Gender', 'ParentalSupport']
Boolean column  : Online Classes Taken

--- Imputing Numeric Columns with TRAIN Median ---
  StudyHoursPerWeek: median = 17.00
  PreviousGrade: median = 78.00
  ExtracurricularActivities: median = 1.00
  Attendance (%): median = 76.00

--- Imputing Categorical Columns with TRAIN Mode ---
  Gender: mode = Male
  ParentalSupport: mode = High

--- Imputing Boolean Column ---
  Online Classes Taken: mode = True

Dropped 32 train rows with missing FinalGrade
Dropped 8  test  rows with missing FinalGrade

Final shapes -- Train: (768, 8), Test: (192, 8)
Remaining missing in train: 0
Remaining missing in test : 0


## Cell 14 — Preprocessing: Encode Categorical Columns

Regression math requires all inputs to be numeric. We encode:

- `Gender` → **Binary encoding**: Male=1, Female=0
- `ParentalSupport` → **Ordinal encoding**: Low=0, Medium=1, High=2  
  *(Ordinal is justified here because parental support has a natural order — more support is better)*
- `Online Classes Taken` → **Boolean to int**: True=1, False=0

In [64]:
# Encode Gender: Male=1, Female=0
gender_map = {'Male': 1, 'Female': 0}
train['Gender'] = train['Gender'].map(gender_map)
test['Gender']  = test['Gender'].map(gender_map)

# Encode ParentalSupport: ordinal (Low < Medium < High)
support_map = {'Low': 0, 'Medium': 1, 'High': 2}
train['ParentalSupport'] = train['ParentalSupport'].map(support_map)
test['ParentalSupport']  = test['ParentalSupport'].map(support_map)

# Encode Online Classes Taken: boolean to integer
train['Online Classes Taken'] = train['Online Classes Taken'].astype(int)
test['Online Classes Taken']  = test['Online Classes Taken'].astype(int)

print('Encoding complete. Sample of encoded columns (train):')
display(train[['Gender', 'ParentalSupport', 'Online Classes Taken']].head(8))

print('\nAll column dtypes after encoding:')
print(train.dtypes)

Encoding complete. Sample of encoded columns (train):


,Gender,ParentalSupport,Online Classes Taken
0,0,0,1
1,0,2,0
2,0,0,1
3,1,2,1
4,0,0,0
5,0,1,0
6,1,2,0
7,1,0,0



All column dtypes after encoding:
Gender                         int64
StudyHoursPerWeek            float64
PreviousGrade                float64
ExtracurricularActivities    float64
ParentalSupport                int64
FinalGrade                   float64
Attendance (%)               float64
Online Classes Taken           int64
dtype: object


## Cell 15 — Separate X and Y

We now separate the feature matrix X from the target vector Y.  
This is the format expected by the modeling team.

In [65]:
# Separate features (X) and target (Y)
TARGET = 'FinalGrade'

X_train = train.drop(columns=[TARGET]).values.astype(float)
Y_train = train[TARGET].values.astype(float)

X_test  = test.drop(columns=[TARGET]).values.astype(float)
Y_test  = test[TARGET].values.astype(float)

feature_names = train.drop(columns=[TARGET]).columns.tolist()

print(f'X_train : {X_train.shape}  ({X_train.shape[0]} samples x {X_train.shape[1]} features)')
print(f'Y_train : {Y_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'Y_test  : {Y_test.shape}')
print(f'\nFeatures: {feature_names}')

X_train : (768, 7)  (768 samples x 7 features)
Y_train : (768,)
X_test  : (192, 7)
Y_test  : (192,)

Features: ['Gender', 'StudyHoursPerWeek', 'PreviousGrade', 'ExtracurricularActivities', 'ParentalSupport', 'Attendance (%)', 'Online Classes Taken']


---
## Cell 16 — Feature Scaling (Z-score Standardization)

### Why scale?
Gradient Descent (implemented by the modeling team) converges much faster when all features are on the same scale.  
Without scaling, `StudyHoursPerWeek` (range ~5–30) would dominate `Gender` (range 0–1).

### Formula:
$$X_{scaled} = \frac{X - \mu_{train}}{\sigma_{train}}$$

After scaling, train features will have **mean ≈ 0** and **std ≈ 1**.

> Again: compute mu and sigma from **train only**, then apply to both sets.

In [66]:
# Z-score standardization using NumPy only -- no sklearn!

# Compute mean and std from TRAIN only
train_mean = X_train.mean(axis=0)   # shape: (n_features,) -- one mean per feature
train_std  = X_train.std(axis=0)    # shape: (n_features,) -- one std per feature

# Protect against division by zero for any constant feature
train_std[train_std == 0] = 1

# Apply standardization: use TRAIN statistics for both sets
X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled  = (X_test  - train_mean) / train_std   # use TRAIN mean/std -- no leakage!

print('Standardization complete.')
print(f'\nX_train_scaled mean per feature (should be ~0):')
print(dict(zip(feature_names, X_train_scaled.mean(axis=0).round(3))))

print(f'\nX_train_scaled std per feature (should be ~1):')
print(dict(zip(feature_names, X_train_scaled.std(axis=0).round(3))))

Standardization complete.

X_train_scaled mean per feature (should be ~0):
{'Gender': np.float64(0.0), 'StudyHoursPerWeek': np.float64(0.0), 'PreviousGrade': np.float64(0.0), 'ExtracurricularActivities': np.float64(0.0), 'ParentalSupport': np.float64(-0.0), 'Attendance (%)': np.float64(0.0), 'Online Classes Taken': np.float64(0.0)}

X_train_scaled std per feature (should be ~1):
{'Gender': np.float64(1.0), 'StudyHoursPerWeek': np.float64(1.0), 'PreviousGrade': np.float64(1.0), 'ExtracurricularActivities': np.float64(1.0), 'ParentalSupport': np.float64(1.0), 'Attendance (%)': np.float64(1.0), 'Online Classes Taken': np.float64(1.0)}


---
## Cell 17 — Save Processed Data for Modeling Team

We save all arrays as `.npz` files so the modeling team can load them directly  
without re-running the preprocessing.

In [67]:
# Save scaled data (use this for gradient descent)
np.savez(
    'processed_data.npz',
    X_train       = X_train_scaled,
    Y_train       = Y_train,
    X_test        = X_test_scaled,
    Y_test        = Y_test,
    feature_names = np.array(feature_names)
)

# Save unscaled data + scaling params (useful for coefficient interpretation)
np.savez(
    'processed_data_unscaled.npz',
    X_train       = X_train,
    Y_train       = Y_train,
    X_test        = X_test,
    Y_test        = Y_test,
    feature_names = np.array(feature_names),
    train_mean    = train_mean,
    train_std     = train_std
)

print('Files saved:')
print('  processed_data.npz          <- scaled  (use for gradient descent modeling)')
print('  processed_data_unscaled.npz <- unscaled + scaling stats')

print('\nHow to load in the modeling notebook:')
print('''
  data = np.load("processed_data.npz", allow_pickle=True)
  X_train = data["X_train"]   # shape: (n_train, n_features)
  Y_train = data["Y_train"]   # shape: (n_train,)
  X_test  = data["X_test"]    # shape: (n_test, n_features)  -- DO NOT USE UNTIL FINAL EVAL
  Y_test  = data["Y_test"]    # shape: (n_test,)             -- DO NOT USE UNTIL FINAL EVAL
''')

Files saved:
  processed_data.npz          <- scaled  (use for gradient descent modeling)
  processed_data_unscaled.npz <- unscaled + scaling stats

How to load in the modeling notebook:

  data = np.load("processed_data.npz", allow_pickle=True)
  X_train = data["X_train"]   # shape: (n_train, n_features)
  Y_train = data["Y_train"]   # shape: (n_train,)
  X_test  = data["X_test"]    # shape: (n_test, n_features)  -- DO NOT USE UNTIL FINAL EVAL
  Y_test  = data["Y_test"]    # shape: (n_test,)             -- DO NOT USE UNTIL FINAL EVAL



In [76]:
df_train_processed = pd.DataFrame(X_train_scaled, columns=feature_names)
df_train_processed['FinalGrade'] = Y_train

df_train_processed.to_csv(r'E:\7777777\train_processed.csv', index=False)

In [ ]:
df_test_processed = pd.DataFrame(X_test_scaled, columns=feature_names)
df_test_processed['FinalGrade'] = Y_test

df_test_processed.to_csv('E:/7777777/test_processed.csv', index=False)

In [80]:
df_train_unscaled = pd.DataFrame(X_train, columns=feature_names)
df_train_unscaled['FinalGrade'] = Y_train

df_train_unscaled.to_csv('E:/7777777/train_unscaled.csv', index=False)



---
## Cell 18 — Final Summary

### What was done in this notebook:

| Step | Action | Justification |
|------|--------|---------------|
| Split | 80/20 train/test (seed=42) | Prevent data leakage; test set locked until final eval |
| EDA | Distribution, correlations, categoricals | On train only; identified strongest predictors |
| Drop | `StudentID`, `Name` | Identifiers, no predictive value |
| Drop redundant | `AttendanceRate`, `Study Hours` | Different scale measurements, keeping cleaner equivalents |
| Fix outlier | Cap `Attendance (%)` at 100 | Domain rule: attendance cannot exceed 100% |
| Impute numeric | Median from train | Robust to outliers; stats from train only |
| Impute categorical | Mode from train | Most frequent is safest assumption; no leakage |
| Encode | Binary + ordinal encoding | Required for regression math |
| Scale | Z-score using train mean/std | Required for gradient descent convergence |

---

### Handoff to Modeling Team:

| File | Contents | Usage |
|------|----------|-------|
| `processed_data.npz` | X_train_scaled, Y_train, X_test_scaled, Y_test | Use for model fitting |
| `processed_data_unscaled.npz` | Raw X, Y + scaling params | For interpretability |

> **IMPORTANT:** Use `X_train` and `Y_train` for all model fitting and K-Fold Cross Validation.  
> **Do NOT touch `X_test` / `Y_test` until the very final evaluation step.**